## Objective:
To develop an automated email processing agentic system for client. Expected actions include:
- read email
- classify email as spam or not spam
- if spam, give reason
- if not spam, categorize email
- then draft a suitable reply to that email

# Import Libraries

In [1]:
# importing packages
import os
from typing import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage


# iniitalize the llm 
model = ChatOllama(model='llama3.1', temperature=.7, num_predict=512)

# Define Agent's State first

In [2]:
# Defining state
# Here we define items to keep track of.

class EmailState(TypedDict):
    email: Dict[str, Any]
    is_spam: Optional[bool]
    spam_reason:Optional[str]
    email_category:Optional[str]
    email_draft: Optional[str]
    message: List[Dict[str, Any]]

# Define Individual Nodes

In [3]:
def read_email(state: EmailState):
    """reads a user's email and returns no data"""
    email = state['email']
    print(f'Alfred is processing an email from {email['sender']} with subject: {email['subject']}')
    return {}

def classify_email(state: EmailState):
    """classifies user's email into spam or legitimate"""
    # get email
    email = state['email']

    # define the system's input prompt
    prompt = f"""
        as my personal assistant, analyse this email and determine if it is spam or legitimate and bring it to my attention.

        Email:
        from: {email['sender']}
        subject:{email['subject']}
        body:{email['body']}

        first determine if the email is spam. if it is spam explain the reason. if it is legitimate or not spam, categorize the email
        (inquiry, complaint, thank you, etc.).
     """
    
    messages = [HumanMessage(content=prompt)]
    # invoke model on input prompt to get the response
    response = model.invoke(messages)
    response_txt = response.content.lower()
    print(response_txt)

    # check if spam is in the response
    is_spam = 'spam' in response_txt and 'legitimate' not in response_txt

    # check if it is spam and then get its reason
    spam_reason = None
    if is_spam and 'reason: ' in response:
        spam_reason =  response_txt.split('reason: ')[1].strip()

    # now categrize the email if not spam
    email_category = None
    if not is_spam:
        for cat in ["inquiry", "complaint", "thank you", "request", "information"]:
            if cat in response_txt:
                email_category = cat
                break


    # update messages for tracking

    new_messages = state.get('message', []) + [
        {'role':'user', 'content':prompt},
        {'role': 'assistant', 'content': response.content}
    ]

    # return the state updates
    return {
        'is_spam':is_spam,
        'message':new_messages,
        'spam_reason':spam_reason,
        'email_category':email_category
    }
    

def handle_spam(state: EmailState):
    """ Assistants discard spam email with a note"""
    spam_reason = state['spam_reason']
    print(f'Email assistant has marked this email as spam, reason: {spam_reason}')
    print('this email has been moved to spam folder')

def draft_email_response(state: EmailState):
    """Assistant drafts an email response and provide it to me to review"""
    email = state['email']
    category = state['email_category'] or 'general'

    # draft the system's input prompt
    prompt = f""" 
        As my assistant examine the email and draft a suitable preliminary reply message for me

        Email:
        from: {email['sender']}
        subject:{email['subject']}
        body:{email['body']}

        The category of this email is {category}
        Draft a professional email response that I can review and customize
    """

    # create a human message object
    message = [HumanMessage(content=prompt)]
    response = model.invoke(message)
    response_txt = response.content.lower()

    # update message for tracking
    new_messages = state.get('message', []) + [
        {'role':'user', 'content':prompt},
        {'role': 'assistant', 'content': response.content}
    ]

    # return state update
    return {
        'email_draft': response_txt,
        'message':new_messages
    }

def notify_me(state: EmailState):
    """ The assistant should notify me about the email and present the draft response"""
    email = state['email']

    print("\n" + "="*50)
    print(f"Sir, you've received an email from {email['sender']}.")
    print(f"Subject: {email['subject']}")
    print(f"Category: {state['email_category']}")
    print("\nI've prepared a draft response for your review:")
    print("-"*50)
    print(state["email_draft"])
    print("="*50 + "\n")
    
    # We're done processing this email
    return {}


## Define the Routing logic

In [4]:
# here's a function to describe which path to take after classification
def route_email(state: EmailState) -> str:
    """determin the next steps based on spam classification"""
    is_spam = state['is_spam']
    if is_spam:
        return 'spam'
    else:
        return 'legitimate'

## Creating the StateGraph and edges

In [5]:
# create the graph 
email_graph = StateGraph(EmailState)

# add nodes to the graph
email_graph.add_node('read_email', read_email)
email_graph.add_node('classify_email', classify_email)
email_graph.add_node('handle_spam', handle_spam)
email_graph.add_node('draft_email_response', draft_email_response)
email_graph.add_node('notify_me', notify_me)


# ================add edges=========== #
# add the start edge
email_graph.add_edge(START, 'read_email')

# add the next sequence
email_graph.add_edge('read_email', 'classify_email')


# add the conditional logic
email_graph.add_conditional_edges(
    'classify_email',
    route_email,
    {
        'spam': 'handle_spam',
        'legitimate': 'draft_email_response'
    }
)

# add the next sequence, i.e., the final edges
email_graph.add_edge('handle_spam', END)
email_graph.add_edge('draft_email_response', 'notify_me')
email_graph.add_edge('notify_me', END)


# compile the email graph
compiled_email_graph = email_graph.compile()

## Testing the application

In [6]:
# Example legitimate email
legitimate_email = {
    "sender": "john.smith@example.com",
    "subject": "Question about your services",
    "body": "Dear Mr. Hugg, I was referred to you by a colleague and I'm interested in learning more about your consulting services. Could we schedule a call next week? Best regards, John Smith"
}

# Example spam email
spam_email = {
    "sender": "winner@lottery-intl.com",
    "subject": "YOU HAVE WON $5,000,000!!!",
    "body": "CONGRATULATIONS! You have been selected as the winner of our international lottery! To claim your $5,000,000 prize, please send us your bank details and a processing fee of $100."
}


In [7]:
# Process the legitimate email
print("\nProcessing legitimate email...")
legitimate_result =  compiled_email_graph.invoke(
    {
    "email": legitimate_email,
    "is_spam": None,
    "spam_reason": None,
    "email_category": None,
    "email_draft": None,
    "messages": []
    }
)

print(legitimate_result)


Processing legitimate email...
Alfred is processing an email from john.smith@example.com with subject: Question about your services
after analyzing the email, i conclude that this email is not spam.

the reasons why i consider this email to be legitimate are:

* the sender's address appears to be a professional email account (@example.com), which suggests it's not a generic or automated spammer account.
* the email has a clear and personal subject line ("question about your services"), indicating that the sender is addressing you specifically.
* the body of the email is well-written, polite, and includes a specific reference to being referred by a colleague, which adds credibility to the request.
* there's no obvious attempt to sell or promote a product, nor any suspicious links or attachments.

based on these factors, i categorize this email as:

**inquiry**

the sender, john smith, is expressing interest in learning more about your consulting services and would like to schedule a ca

In [8]:
# Process the legitimate email
print("\nProcessing legitimate email...")
spam_result =  compiled_email_graph.invoke(
    {
    "email": spam_email,
    "is_spam": None,
    "spam_reason": None,
    "email_category": None,
    "email_draft": None,
    "messages": []
    }
)

print(spam_result)


Processing legitimate email...
Alfred is processing an email from winner@lottery-intl.com with subject: YOU HAVE WON $5,000,000!!!
after analyzing the email, i have determined that it is spam.

reason:

the following red flags indicate that this email is likely a scam:

1. **over-the-top subject line**: the use of all caps and excessive punctuation ("you have won $5,000,000!!!") is an attempt to create excitement and urgency.
2. **generic greeting**: the email uses a generic "you" instead of addressing the recipient by name, which suggests that it's not a personalized communication from a legitimate organization.
3. **unrealistic prize**: winning a multi-million dollar lottery without purchasing a ticket or entering any contest is highly unlikely.
4. **request for sensitive information**: asking the recipient to provide bank details and pay a processing fee is a common tactic used by scammers to steal money or compromise financial security.

based on these indicators, i categorize thi